# C10 prereg68 X2：prereg61 配方、種子 1、400,000 步（Colab G4）

執行階段選 G4（執行階段版本 2026.07），按「執行階段 → 全部執行」。不用改任何程式。
套件共用：C10/p42 kit、C10/p51/overlay_parts、C10/p62/overlay2_parts。
輸出在 C10/p62/X2/，每 5,000 步存檔，斷線後重按「全部執行」會接續（約 22 小時；24 小時上限前若斷線，重開執行階段再全部執行）。
練完打包 C10/p62/C10_p62_X2_results.zip。任何一格出現紅字或 STOP，停下來，截最後 20 行。

In [ ]:
# 0. 把 Drive 上 C10/p42/kit_parts/ 的 66 份接回 C10_p42_kit.zip（已存在且雜湊對就略過）
import hashlib, os, glob
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/C10/p42'
KIT = os.path.join(D, 'C10_p42_kit.zip')
SHA = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
if not (os.path.exists(KIT) and sha(KIT) == SHA):
    parts = sorted(glob.glob(os.path.join(D, 'kit_parts', 'kit.part*')))
    assert len(parts) == 66, 'STOP: kit_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p42_kit.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == SHA, 'STOP: 接回的 zip 雜湊不對'
    import shutil; shutil.copyfile(tmp, KIT)
    assert sha(KIT) == SHA, 'STOP: 複製到 Drive 後雜湊不對'
print('kit OK', SHA[:12])


In [ ]:
# 1. 雲端硬碟、GPU、套件（不用改）
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE, WORK, VARIANT = '/content/drive/MyDrive/C10/p42', '/content/c10', 'V2'
KIT_SHA, FILES = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306', {"data/prereg06/feats/meta.npz": "b1b5def7312dc6321ab08dd7e74e9b9069c49e9efb8caf7133b9869bf7338df5", "data/prereg36/cache/cache.json": "764a61937e60ab660972ecbe8a585e4205bd27a132ce74a64f4cc7390533c48a", "data/prereg36/cache/codes.npy": "be5831abbccf22be3c96725082939638e6607f88ad45b1ed2164db33060c0303", "data/prereg36/cache/meta.npz": "fa8bf685be64aa100746238caeaa87f917d6adc194d2026d83c7865f61f834d3", "data/prereg36/cache/off.npy": "7c352279359c5035c1693b78a783d534c8b6ac1e7f870e56655b4b764fb2eaf8", "results/prereg36/explore/p36_embed.py": "2f2e732d1309e25e0b70e370583c07b74dd4c692c6490db8fe1b533ecda32c21", "results/prereg36/explore/p36_lib.py": "8df27b37a6b3de786b54af25908d3d91ffe4c030dceb7350d9dd130a14ce241f", "results/prereg36/explore/p36_train.py": "8abc5ef22eba35b70be06c624f7071218d0a4266affe5494d1207f8819271211", "results/prereg36/explore/split_record.json": "2c847257139cd73b9a1808c003ba30b86bcd59eb6860edef8043afc82535e7e3", "results/prereg42/explore/p42_embed.py": "23b60a6f71d1aad5881acd10c87ef2c31a9ed81e6ac2d369c3d29615452b69fd", "results/prereg42/explore/p42_lib.py": "b5045f8b8222e1c3e6891675eb4b3296fe4254b974421c71853ea8a43cea6eea", "results/prereg42/explore/p42_train.py": "ec5d7baf7bf16753cd9d0fb7369e3ff4cb5a9a271d8ed28cf264691bb2359c80", "tools/c10data.py": "2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()
print('torch', tv)   # in a subprocess: the notebook itself never holds the GPU (p42_train refuses to share it)
assert tv.endswith('True'), 'STOP: 沒有 GPU，請在「變更執行階段類型」選 GPU'
kit = os.path.join(DRIVE, 'C10_p42_kit.zip')
assert os.path.exists(kit), 'STOP: 找不到 ' + kit
ok_flag = os.path.join(WORK, '.kit_ok')
if not (os.path.exists(ok_flag) and open(ok_flag).read() == KIT_SHA):
    assert sha(kit) == KIT_SHA, 'STOP: 套件檔的 SHA-256 不對'
    with zipfile.ZipFile(kit) as z:
        z.extractall(WORK)
    for rel, h in FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(ok_flag, 'w').write(KIT_SHA)
print('kit ok', len(FILES), 'files')
os.makedirs(os.path.join(DRIVE, VARIANT), exist_ok=True)

In [ ]:
# 1b. prereg51 的程式和 T49 快取：從 C10/p51/overlay_parts/ 接回 overlay zip、核雜湊、解到工作目錄（不用改）
import glob, shutil
P51 = '/content/drive/MyDrive/C10/p51'
OSHA, NPARTS, OFILES = '2729cbb3cf022817a84b217d5cb33a567e63eb5a4c4e55b50e3f69fb2c23f654', 77, {"data/prereg49/cache/cache.json": "b38714742e93787a659932e10cac50ab208813dcfeae1fd82f472c1e45fc9219", "data/prereg49/cache/codes.npy": "2447f0e05a42c3634a3ba16ba2666affeed95e90c4c8724ed29d8a4ee849a09a", "data/prereg49/cache/meta.npz": "c6821af3c8752fb77e1983466ce658d99046669e1bc2b2e4f37ecc9b7bb8ad34", "data/prereg49/cache/off.npy": "f4db9a7caf02fdec324414f7aee6a64ebdc52238257f9c28bda02a468a3553a0", "results/prereg43/explore/p43_run.py": "a767ad139e921de3ff5fe2afe48165c048efa2afdf944bc8402e9ec300f98064", "results/prereg49/explore/p49_lib.py": "3f2b48c0e078cf5d87f6856ab3618b9de31fada8723682868e6ebed2c817c631", "results/prereg49/explore/p49_run.py": "674936efb925fc65beb6f53ac0434473be2b71810d0e408c4cb084dac3617fb2", "results/prereg50/explore/p50_run.py": "12017e38d81097e87bb6f3a778f421ec978e35794af7e1aac8dd9bcc38ef20c9", "results/prereg51/explore/p51_run.py": "7495c9cd31fa3b01ec884235a09c5c5ed38f16a1635b9d10000786d51eb860b7"}
flag = os.path.join(WORK, '.p51_ok')
if not (os.path.exists(flag) and open(flag).read() == OSHA):
    parts = sorted(glob.glob(os.path.join(P51, 'overlay_parts', 'overlay.part*')))
    assert len(parts) == NPARTS, 'STOP: overlay_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p51_overlay.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == OSHA, 'STOP: 接回的 overlay zip 雜湊不對'
    with zipfile.ZipFile(tmp) as z:
        z.extractall(WORK)
    for rel, h in OFILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(flag, 'w').write(OSHA)
print('overlay ok', OSHA[:12])

In [ ]:
# 1c. prereg60／61 的程式和 c49 存檔：從 C10/p62/overlay2_parts/ 接回 overlay2 zip、核雜湊、解到工作目錄（不用改）
P62 = '/content/drive/MyDrive/C10/p62'
O2SHA, N2PARTS, O2FILES = 'f9f7416f7cd7fe236344c2ff7bba3112e23790e9b555843a87a6113a37155cbd', 10, {"data/prereg49/S0/ckpt/step_320000.pt": "c091d3af1797a19ce98dc0a670eea211ceca2de4b58ee4a3c62b73b854748e12", "results/prereg60/explore/p60_lib.py": "431e2b5276470a59644d2d3226ff9f06cfe4f2e18a6dfedb51f6bd10c8b21f23", "results/prereg60/explore/p60_run.py": "d87ae4b63ca241267213f6d17aa6dd25e6e42a1011db17111ff7d430f23ffc1a", "results/prereg61/explore/p61_run.py": "654f6f7a55cd2f83062e2c418fd25f4db7b7e6a268086d63711d8670bc3c6d63"}
flag = os.path.join(WORK, '.p62_ok')
if not (os.path.exists(flag) and open(flag).read() == O2SHA):
    parts = sorted(glob.glob(os.path.join(P62, 'overlay2_parts', 'overlay2.part*')))
    assert len(parts) == N2PARTS, 'STOP: overlay2_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p62_overlay2.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == O2SHA, 'STOP: 接回的 overlay2 zip 雜湊不對'
    with zipfile.ZipFile(tmp) as z:
        z.extractall(WORK)
    for rel, h in O2FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(flag, 'w').write(O2SHA)
print('overlay2 ok', O2SHA[:12])

In [ ]:
# 1d. prereg62 的包裝程式 p62_run.py（--arm 決定種子和步數）；核雜湊後寫進工作目錄（不用改）
import base64
ARM, LAST = 'X2', 400000
H, B64 = 'a563aadb6c0ca325724f94581a28a80db87f4c6e9ce2faa7f654d8b8d4534ea6', 'IiIicHJlcmVnNjIgKFBMQU4ubWQpOiBwcmVyZWc2MSdzIHJlY2lwZSwgYXJtcyBTMSAoc2VlZCAxKSwgUzIgKHNlZWQgMiksIFggKHNlZWQgMCwgNDAwLDAwMCBzdGVwcyksIGZvciBDb2xhYiBHNC4KQSB3cmFwcGVyIG92ZXIgcDYxX3J1biAod2hpY2ggd3JhcHMgcDYwX3J1bik7IG5vdGhpbmcgb2YgcHJlcmVnNjAgLyBwcmVyZWc2MSBpcyBlZGl0ZWQuCgogICAgUFkgcmVzdWx0cy9wcmVyZWc2Mi9leHBsb3JlL3A2Ml9ydW4ucHkgdHJhaW4gLS1hcm0gUzF8UzJ8WCBbLS13b3JrZXJzIE5dIFstLWF1dG8tcmVzdW1lXSBbLS1vdXQgRElSXSBbLS1yZWNvcmQgRl0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIFstLW1pbi1mcmVlLXJhbS1tYiAyMDQ4XSBbLS1sb2ctZXZlcnkgMTAwMF0KICAgIFBZIHJlc3VsdHMvcHJlcmVnNjIvZXhwbG9yZS9wNjJfcnVuLnB5IHRyYWluIC0tYXJtIFMxIC0tc21va2UgLS1kZXZpY2UgY3B1IC0td29ya2VycyAwIFstLXN0ZXBzIE48PTNdIFstLWF1dG8tcmVzdW1lXQogICAgUFkgcmVzdWx0cy9wcmVyZWc2Mi9leHBsb3JlL3A2Ml9ydW4ucHkgZW1iZWQgLS1hcm0gUzEgLS1ja3B0IGRhdGEvcHJlcmVnNjIvUzEvY2twdC9zdGVwXzIwMDAwMC5wdCAtLXNvdXJjZSBzdG9yZSAtLXBsYXllcnMgRQoKQW4gYXJtIGZpeGVzIHRoZSBzZWVkIGFuZCB0aGUgbnVtYmVyIG9mIHN0ZXBzOyBub3RoaW5nIGVsc2UgZGlmZmVycyAoUExBTi5tZCB0YWJsZSk6CgogICAgUzE6IHNlZWQgMSwgMjAwLDAwMCBzdGVwcywga2VwdCA1MGsvMTAway8xNTBrLzIwMGsgICAgICBTMjogc2VlZCAyLCB0aGUgc2FtZQogICAgWCA6IHNlZWQgMCwgNDAwLDAwMCBzdGVwcywga2VwdCAxMDBrLzIwMGsvMzAway80MDBrCgpXaGF0IHRoaXMgd3JhcHBlciBzZXRzIG9uIHA2MF9ydW4gYWZ0ZXIgcDYxX3J1bi5jb25maWd1cmUoKSAobHIgMWUtMywgd2FybS11cCAyLDAwMCBzdGF5IHByZXJlZzYxJ3MpOgoKICBEQVRBIC8gT1VUIC8gU1RPUF9GSUxFIC0+IGRhdGEvcHJlcmVnNjIvPGFybT4gKG9yIC0tb3V0OiB0aGUgQ29sYWIgRHJpdmUgZm9sZGVyKSwgRU1CIC8gU0VBTSAvIFNFQU1fR1BVIHVuZGVyIERBVEEsCiAgVFJBSU5fUkVDT1JEIC0+IHJlc3VsdHMvcHJlcmVnNjIvZXhwbG9yZS90cmFpbl9yZWNvcmRfPGFybT4uanNvbiAob3IgLS1yZWNvcmQpLAogIFNFRUQsIFRPVEFMX1NURVBTLCBLRUVQIHBlciBhcm07IFJFU1VNRV9FVkVSWSA1LDAwMCAoQ29sYWI6IDw9IH4yMCBtaW4gb2Ygd29yayBsb3N0KTsKICBDT0RFX0ZJTEVTICsgdGhpcyBmaWxlOwogIHA2MF9saWIucHJ1bmUgLT4gcHJ1bmUyOiBrZWVwcyB0aGUga2VwdCBzdGVwcyBBTkQgdGhlIHR3byBtb3N0IHJlY2VudCBjaGVja3BvaW50cyAodGhlIG5ld2VzdCByZXN1bWFibGUgb25lIGlzCiAgbmV2ZXIgdGhlIG9ubHkgY29weSB3aGlsZSBEcml2ZSBpcyBzeW5jaW5nKTsKICBiZWZvcmUgYSB0cmFpbmluZyBydW46IGEgbmV3ZXN0IGNoZWNrcG9pbnQgdGhhdCB0b3JjaCBjYW5ub3QgcmVhZCBpcyByZW5hbWVkIDxuYW1lPi5iYWQsIHNvIHRoYXQgcmVzdW1lIChoaWdoZXN0CiAgc3RlcCBudW1iZXIsIHA2MF9saWIubGFzdF9ja3B0KSB0YWtlcyB0aGUgb25lIGJlZm9yZTsKICBzZWVkICE9IDAgb25seTogcDYwX3J1biBjb21wYXJlcyBpdHMgU0VFRCB3aXRoIHRoZSBzZWVkIHN0b3JlZCBpbiB0aGUgYzQ5IGNoZWNrcG9pbnQgKDApIGluc2lkZSBpdHMgImlzIHRoaXMKICBjNDkncyByZWNpcGUiIGNoZWNrLiB0b3JjaC5sb2FkIGlzIHdyYXBwZWQgZm9yIHRoYXQgb25lIGZpbGU6IHRoZSB3cmFwcGVyIGNoZWNrcyBpdHNlbGYgdGhhdCB0aGUgc3RvcmVkIHNlZWQgaXMKICAwIGFuZCB0aGVuIGhhbmRzIHA2MF9ydW4gYSBjb25maWcgd2hvc2Ugc2VlZCBpcyB0aGUgYXJtJ3MsIHNvIHRoYXQgc2luZ2xlIGNvbXBhcmlzb24gcGFzc2VzIGFuZCBhbGwgdGhlIG90aGVycwogICh2YXJpYW50LCBzcGVjLCBibG9ja3MsIGNoYW5uZWxzLCBlbWJfZGltLCBzY2FsZSwgd2QsIEVNQSBkZWNheSwgYmF0Y2ggc2hhcGU7IGNsYXNzZXMsIGNhY2hlIGhhc2ggYW5kIHRoZQogIGNoZWNrcG9pbnQncyBzaGEyNTYgZnVydGhlciBkb3duKSBhcmUgcDYwX3J1bidzLCB1bnRvdWNoZWQuCgpUaGUgc2VlZCBjb250cm9scyAoUExBTi5tZCk6IHRoZSBzYW1wbGVyIHN0cmVhbSBudW1weS5yYW5kb20uZGVmYXVsdF9ybmcoW3NlZWQsIDMyMDAwMCArIHNdKSBvZiBiYXRjaCBzOwp0b3JjaC5tYW51YWxfc2VlZCAobm8gcmFuZG9tIG9wZXJhdGlvbiBpbiB0aGUgVjIgZm9yd2FyZCBwYXNzIG9yIGxvc3M7IGFsbCB3ZWlnaHRzIGNvbWUgZnJvbSBjNDkpOyB0aGUgbnVtYmVyCndyaXR0ZW4gaW4gdGhlIGNvbmZpZy4gQ2hlY2twb2ludHMgY2FycnkgcHJlcmVnNjAncyBjb25maWcga2V5ICJwNjAiOyB0aGUgdmVjdG9ycycgdGFnIGlzCnA2MF9zdGVwXzxzdGVwcz5fcmF3X3R0YTEgdW5kZXIgZGF0YS9wcmVyZWc2Mi88YXJtPi9lbWIuCgpObyBzY29yZSBvZiBhbnkga2luZCBpcyBjb21wdXRlZCBoZXJlLiBOb3RoaW5nIGhlcmUgb3BlbnMgdGhlIHZhbGlkYXRpb24tYW5zd2VyIGZvbGRlciBvciBydW5zIHRvb2xzL3Njb3JlLnB5LgoiIiIKaW1wb3J0IGdsb2IKaW1wb3J0IG9zCmltcG9ydCBzeXMKCkhFUkUgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKUkVQTyA9IG9zLnBhdGguYWJzcGF0aChvcy5wYXRoLmpvaW4oSEVSRSwgIi4uIiwgIi4uIiwgIi4uIikpClA2MUQgPSBvcy5wYXRoLmpvaW4oUkVQTywgInJlc3VsdHMiLCAicHJlcmVnNjEiLCAiZXhwbG9yZSIpCmlmIFA2MUQgbm90IGluIHN5cy5wYXRoOgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIFA2MUQpCmltcG9ydCBwNjFfcnVuIGFzIFc2MSAgIyBub3FhOiBFNDAyCgpSLCBQID0gVzYxLlIsIFc2MS5QCkFSTVMgPSB7IlMxIjogeyJzZWVkIjogMSwgInN0ZXBzIjogMjAwMDAwLCAia2VlcCI6ICg1MDAwMCwgMTAwMDAwLCAxNTAwMDAsIDIwMDAwMCl9LAogICAgICAgICJTMiI6IHsic2VlZCI6IDIsICJzdGVwcyI6IDIwMDAwMCwgImtlZXAiOiAoNTAwMDAsIDEwMDAwMCwgMTUwMDAwLCAyMDAwMDApfSwKICAgICAgICAiWCI6IHsic2VlZCI6IDAsICJzdGVwcyI6IDQwMDAwMCwgImtlZXAiOiAoMTAwMDAwLCAyMDAwMDAsIDMwMDAwMCwgNDAwMDAwKX19ClJFU1VNRV9FVkVSWSA9IDUwMDAKUkVDRU5UID0gMgpDNDlfU0VFRCA9IDAKCgpkZWYgcG9wKHJlc3QsIGZsYWcsIGRlZmF1bHQ9Tm9uZSk6CiAgICBpZiBmbGFnIG5vdCBpbiByZXN0OgogICAgICAgIHJldHVybiBkZWZhdWx0CiAgICBpID0gcmVzdC5pbmRleChmbGFnKQogICAgdiA9IHJlc3RbaSArIDFdCiAgICBkZWwgcmVzdFtpOmkgKyAyXQogICAgcmV0dXJuIHYKCgpkZWYgcHJ1bmUyKGQsIGtlZXBfc3RlcHMsIGxhdGVzdCk6CiAgICAiIiJwNjBfbGliLnBydW5lLCBidXQgdGhlIFJFQ0VOVCAoMikgaGlnaGVzdCBzdGVwIG51bWJlcnMgcHJlc2VudCBhcmUga2VwdCBhcyB3ZWxsIGFzIGtlZXBfc3RlcHMgYW5kIGBsYXRlc3RgLiIiIgogICAgc3RlcHMgPSBzb3J0ZWQoKFAuc3RlcF9vZihwKSwgcCkgZm9yIHAgaW4gZ2xvYi5nbG9iKG9zLnBhdGguam9pbihkLCAic3RlcF8qLnB0IikpKQogICAgcmVjZW50ID0ge3MgZm9yIHMsIF8gaW4gc3RlcHNbLVJFQ0VOVDpdfQogICAgZ29uZSA9IFtdCiAgICBmb3IgcywgcCBpbiBzdGVwczoKICAgICAgICBpZiBzIG5vdCBpbiBrZWVwX3N0ZXBzIGFuZCBzICE9IGxhdGVzdCBhbmQgcyBub3QgaW4gcmVjZW50OgogICAgICAgICAgICBvcy5yZW1vdmUocCkKICAgICAgICAgICAgZ29uZS5hcHBlbmQob3MucGF0aC5iYXNlbmFtZShwKSkKICAgIHJldHVybiBzb3J0ZWQoZ29uZSkKCgpkZWYgc2V0X2FzaWRlX3VucmVhZGFibGUoY2twdF9kaXIpOgogICAgIiIiUmVuYW1lIGEgbmV3ZXN0IGNoZWNrcG9pbnQgdGhhdCBjYW5ub3QgYmUgcmVhZCB0byA8bmFtZT4uYmFkIChyZXBlYXRlZGx5KS4gUmV0dXJucyB0aGUgcmVuYW1lZCBuYW1lcy4iIiIKICAgIGltcG9ydCB0b3JjaAogICAgYmFkID0gW10KICAgIHdoaWxlIG9zLnBhdGguaXNkaXIoY2twdF9kaXIpOgogICAgICAgIGxhc3QgPSBQLmxhc3RfY2twdChja3B0X2RpcikKICAgICAgICBpZiBsYXN0IGlzIE5vbmU6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgdHJ5OgogICAgICAgICAgICBjayA9IHRvcmNoLmxvYWQobGFzdCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgICAgIG9rID0gaXNpbnN0YW5jZShjaywgZGljdCkgYW5kICJuZXQiIGluIGNrIGFuZCAic3RlcCIgaW4gY2sKICAgICAgICAgICAgZGVsIGNrCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBub3FhOiBCTEUwMDEgICh0cnVuY2F0ZWQgLyBoYWxmLXN5bmNlZCBmaWxlKQogICAgICAgICAgICBwcmludCgiY2Fubm90IHJlYWQgJXM6ICVzIiAlIChsYXN0LCBzdHIoZSlbOjIwMF0pLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBvayA9IEZhbHNlCiAgICAgICAgaWYgb2s6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgb3MucmVwbGFjZShsYXN0LCBsYXN0ICsgIi5iYWQiKQogICAgICAgIGJhZC5hcHBlbmQob3MucGF0aC5iYXNlbmFtZShsYXN0KSkKICAgICAgICBwcmludCgic2V0IGFzaWRlICVzLmJhZDsgcmVzdW1pbmcgZnJvbSB0aGUgY2hlY2twb2ludCBiZWZvcmUgaXQiICUgb3MucGF0aC5iYXNlbmFtZShsYXN0KSwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBiYWQKCgpkZWYgcGFzc19iYXNlX3NlZWQoc2VlZCk6CiAgICAiIiJTZWUgdGhlIG1vZHVsZSBkb2NzdHJpbmc6IG9ubHkgZm9yIHNlZWQgIT0gMCwgb25seSBmb3IgdGhlIGM0OSBjaGVja3BvaW50IGZpbGUuIiIiCiAgICBpZiBzZWVkID09IEM0OV9TRUVEOgogICAgICAgIHJldHVybgogICAgaW1wb3J0IHRvcmNoCiAgICBvcmlnID0gdG9yY2gubG9hZAogICAgaWYgZ2V0YXR0cihvcmlnLCAiX3A2MiIsIEZhbHNlKToKICAgICAgICByZXR1cm4KICAgIGJhc2UgPSBvcy5wYXRoLmFic3BhdGgoUi5DNDlfQ0tQVCkKCiAgICBkZWYgbG9hZChmLCAqYSwgKiprKToKICAgICAgICBjayA9IG9yaWcoZiwgKmEsICoqaykKICAgICAgICBpZiBpc2luc3RhbmNlKGYsIHN0cikgYW5kIG9zLnBhdGguYWJzcGF0aChmKSA9PSBiYXNlOgogICAgICAgICAgICBpZiBja1siY29uZmlnIl0uZ2V0KCJzZWVkIikgIT0gQzQ5X1NFRUQ6CiAgICAgICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJ0aGUgYzQ5IGNoZWNrcG9pbnQncyBzdG9yZWQgc2VlZCBpcyBub3QgJWQiICUgQzQ5X1NFRUQpCiAgICAgICAgICAgIGNrWyJjb25maWciXSA9IGRpY3QoY2tbImNvbmZpZyJdLCBzZWVkPXNlZWQpCiAgICAgICAgcmV0dXJuIGNrCgogICAgbG9hZC5fcDYyID0gVHJ1ZQogICAgdG9yY2gubG9hZCA9IGxvYWQKCgpkZWYgY29uZmlndXJlKGFybSwgb3V0PU5vbmUsIHJlY29yZD1Ob25lKToKICAgIGEgPSBBUk1TW2FybV0KICAgIFc2MS5jb25maWd1cmUoKQogICAgZGF0YSA9IG9zLnBhdGguam9pbihSRVBPLCAiZGF0YSIsICJwcmVyZWc2MiIsIGFybSkKICAgIFIuREFUQSA9IGRhdGEKICAgIFIuT1VUID0gb3MucGF0aC5hYnNwYXRoKG91dCkgaWYgb3V0IGVsc2UgZGF0YQogICAgUi5FTUIgPSBvcy5wYXRoLmpvaW4oZGF0YSwgImVtYiIpCiAgICBSLlNFQU0gPSBvcy5wYXRoLmpvaW4oZGF0YSwgInNlYW0iKQogICAgUi5TRUFNX0dQVSA9IG9zLnBhdGguam9pbihkYXRhLCAic2VhbV9ncHUiKQogICAgUi5TVE9QX0ZJTEUgPSBvcy5wYXRoLmpvaW4oZGF0YSwgIlNUT1AiKQogICAgUi5UUkFJTl9SRUNPUkQgPSBvcy5wYXRoLmFic3BhdGgocmVjb3JkKSBpZiByZWNvcmQgZWxzZSBvcy5wYXRoLmpvaW4oSEVSRSwgInRyYWluX3JlY29yZF8lcy5qc29uIiAlIGFybSkKICAgIFIuU0VFRCA9IGFbInNlZWQiXQogICAgUi5UT1RBTF9TVEVQUyA9IGFbInN0ZXBzIl0KICAgIFIuS0VFUCA9IGFbImtlZXAiXQogICAgUi5SRVNVTUVfRVZFUlkgPSBSRVNVTUVfRVZFUlkKICAgIFIuZW1iX3RhZyA9IGxhbWJkYSBzdGVwPWFbInN0ZXBzIl06ICJwNjBfc3RlcF8lMDZkX3Jhd190dGExIiAlIGludChzdGVwKQogICAgbWUgPSBvcy5wYXRoLmFic3BhdGgoX19maWxlX18pCiAgICBpZiBtZSBub3QgaW4gUi5DT0RFX0ZJTEVTOgogICAgICAgIFIuQ09ERV9GSUxFUyA9IHR1cGxlKFIuQ09ERV9GSUxFUykgKyAobWUsKQogICAgUC5wcnVuZSA9IHBydW5lMgogICAgcmV0dXJuIFIKCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgYXJndiA9IGxpc3Qoc3lzLmFyZ3ZbMTpdIGlmIGFyZ3YgaXMgTm9uZSBlbHNlIGFyZ3YpCiAgICBpZiBub3QgYXJndiBvciBhcmd2WzBdIG5vdCBpbiAoInRyYWluIiwgImVtYmVkIik6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgidXNhZ2U6IHA2Ml9ydW4ucHkgdHJhaW58ZW1iZWQgLS1hcm0gUzF8UzJ8WCAuLi4iKQogICAgcmVzdCA9IGFyZ3ZbMTpdCiAgICBhcm0gPSBwb3AocmVzdCwgIi0tYXJtIikKICAgIGlmIGFybSBub3QgaW4gQVJNUzoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCItLWFybSBtdXN0IGJlIG9uZSBvZiAlcyIgJSBzb3J0ZWQoQVJNUykpCiAgICBvdXQsIHJlY29yZCA9IHBvcChyZXN0LCAiLS1vdXQiKSwgcG9wKHJlc3QsICItLXJlY29yZCIpCiAgICBpZiBhcmd2WzBdID09ICJlbWJlZCI6CiAgICAgICAgaWYgb3V0IG9yIHJlY29yZDoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1vdXQgLyAtLXJlY29yZCBhcmUgZm9yIHRyYWluIikKICAgICAgICBjb25maWd1cmUoYXJtKQogICAgICAgIHJldHVybiBSLm1haW4oWyJlbWJlZCJdICsgcmVzdCkKICAgIGNvbmZpZ3VyZShhcm0sIG91dCwgcmVjb3JkKQogICAgcHJpbnQoInByZXJlZzYyIGFybSAlczogc2VlZCAlZCwgJWQgc3RlcHMsIGxyICVnLCB3YXJtLXVwICVkLCBrZWVwICVzLCBvdXQgJXMiICUgKAogICAgICAgIGFybSwgUi5TRUVELCBSLlRPVEFMX1NURVBTLCBSLkxSLCBSLldBUk1VUCwgbGlzdChSLktFRVApLCBSLk9VVCksIGZsdXNoPVRydWUpCiAgICBydW5fb3V0ID0gb3MucGF0aC5qb2luKFIuREFUQSwgInNtb2tlIiwgIlMwIikgaWYgIi0tc21va2UiIGluIHJlc3QgZWxzZSBSLk9VVAogICAgc2V0X2FzaWRlX3VucmVhZGFibGUob3MucGF0aC5qb2luKHJ1bl9vdXQsICJja3B0IikpCiAgICBwYXNzX2Jhc2Vfc2VlZChSLlNFRUQpCiAgICByZXR1cm4gUi5tYWluKFsidHJhaW4iXSArIHJlc3QpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK'
data = base64.b64decode(B64)
assert hashlib.sha256(data).hexdigest() == H, 'STOP: 內嵌的 p62_run.py 不對'
os.makedirs(os.path.join(WORK, 'results/prereg62/explore'), exist_ok=True)
with open(os.path.join(WORK, 'results/prereg62/explore/p62_run.py'), 'wb') as f:
    f.write(data)
out = os.path.join(P62, ARM)
os.makedirs(out, exist_ok=True)
print('p62_run ok', H[:12], 'arm', ARM)
# prereg68 wrapper (adds arms X2 / X3)
WH, WB64 = '9a338af823c9a24bea6fb6a28d3c86251f720648064dfab8ae167a471112ef50', 'IiIicHJlcmVnNjg6IHA2Ml9ydW4gd2l0aCB0d28gbW9yZSBhcm1zIChwcmVyZWc2MSByZWNpcGUsIDQwMCwwMDAgc3RlcHMpOiBYMiA9IHNlZWQgMSwgWDMgPSBzZWVkIDIuCgogICAgUFkgcmVzdWx0cy9wcmVyZWc2OC9leHBsb3JlL3A2OF9jb2xhYl9ydW4ucHkgdHJhaW4gLS1hcm0gWDJ8WDMgLS1hdXRvLXJlc3VtZSAtLW91dCBESVIgLS1yZWNvcmQgRklMRSBbLS13b3JrZXJzIE5dCiAgICBQWSByZXN1bHRzL3ByZXJlZzY4L2V4cGxvcmUvcDY4X2NvbGFiX3J1bi5weSBlbWJlZCAtLWFybSBYMnxYMyAtLWNrcHQgLi4uIFtwNDJfZW1iZWQgYXJndW1lbnRzXQoiIiIKaW1wb3J0IG9zCmltcG9ydCBzeXMKCkhFUkUgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKUkVQTyA9IG9zLnBhdGguYWJzcGF0aChvcy5wYXRoLmpvaW4oSEVSRSwgIi4uIiwgIi4uIiwgIi4uIikpCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmpvaW4oUkVQTywgInJlc3VsdHMiLCAicHJlcmVnNjIiLCAiZXhwbG9yZSIpKQppbXBvcnQgcDYyX3J1biAgIyBub3FhOiBFNDAyCgpLRUVQID0gKDEwMDAwMCwgMjAwMDAwLCAzMDAwMDAsIDQwMDAwMCkKcDYyX3J1bi5BUk1TWyJYMiJdID0geyJzZWVkIjogMSwgInN0ZXBzIjogNDAwMDAwLCAia2VlcCI6IEtFRVB9CnA2Ml9ydW4uQVJNU1siWDMiXSA9IHsic2VlZCI6IDIsICJzdGVwcyI6IDQwMDAwMCwgImtlZXAiOiBLRUVQfQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KHA2Ml9ydW4ubWFpbigpKQo='
wd = base64.b64decode(WB64)
assert hashlib.sha256(wd).hexdigest() == WH, 'STOP: p68_colab_run.py 不對'
os.makedirs(os.path.join(WORK, 'results/prereg68/explore'), exist_ok=True)
with open(os.path.join(WORK, 'results/prereg68/explore/p68_colab_run.py'), 'wb') as f:
    f.write(wd)
print('p68_colab_run ok', WH[:12])

In [ ]:
# 1e. 版本（只印出來；在子程序裡查，筆記本本身不佔 GPU）
print(subprocess.run([sys.executable, '-c', 'import torch; print("torch", torch.__version__, "cuDNN", torch.backends.cudnn.version(), torch.cuda.get_device_name(0))'], capture_output=True, text=True).stdout.strip())
print(subprocess.run('pip list 2>/dev/null | grep -iE "^torch |nvidia-cudnn|^numpy "', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. 訓練 X（約 21 小時；每 5,000 步存到 C10/p62/X/ckpt/；斷線後重按「全部執行」會接續）
import threading, time
workers = str(max(2, min(12, (os.cpu_count() or 4) - 1)))
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PYTHONIOENCODING='utf-8')
def run(args, log, jl):
    with open(log, 'a') as lf:
        p = subprocess.Popen([sys.executable, '-u'] + args, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        def tail():   # 每分鐘把訓練紀錄的最新一行印出來（steps_per_s＝速度）
            seen = None
            while p.poll() is None:
                time.sleep(60)
                try:
                    with open(jl) as f:
                        last = f.read().strip().split('\n')[-1]
                except OSError:
                    continue
                if last and last != seen:
                    seen = last
                    print(last, flush=True)
        threading.Thread(target=tail, daemon=True).start()
        for line in p.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        return p.wait()
rec = os.path.join(out, 'train_record_%s.json' % ARM)
for attempt in range(6):
    if os.path.exists(rec) and json.load(open(rec)).get('finished'):
        break
    assert not os.path.exists(os.path.join(out, 'STOP')), 'STOP: C10/p62/%s/STOP 存在，拿掉後再按全部執行' % ARM
    rc = run(['results/prereg68/explore/p68_colab_run.py', 'train', '--arm', ARM, '--workers', workers, '--auto-resume', '--out', out, '--record', rec, '--min-free-ram-mb', '2048', '--log-every', '500'], os.path.join(out, 'train_log.txt'), os.path.join(out, 'train_log.jsonl'))
    assert rc in (0, 4), 'STOP: 訓練結束碼 %d' % rc
    if rc == 4:
        print('結束碼 4（記憶體不足或 STOP 檔）：60 秒後從存檔接續，第 %d 次' % (attempt + 1))
        time.sleep(60)
assert os.path.exists(rec) and json.load(open(rec)).get('finished'), 'STOP: 還沒練完，重按「全部執行」會接續'
print('training finished')

In [ ]:
# 3. 打包（不用改）
ck = os.path.join(out, 'ckpt', 'step_%06d.pt' % LAST)
res = os.path.join(P62, 'C10_p62_%s_results.zip' % ARM)
with zipfile.ZipFile(res + '.tmp', 'w', zipfile.ZIP_STORED) as z:
    for name in ('train_record_%s.json' % ARM, 'train_log.txt', 'train_log.jsonl'):
        p = os.path.join(out, name)
        if os.path.exists(p):
            z.write(p, '%s/%s' % (ARM, name))
    z.write(ck, '%s/ckpt/step_%06d.pt' % (ARM, LAST))
os.replace(res + '.tmp', res)
drive.flush_and_unmount()
print('完成', os.path.basename(res))